# Experimento de ML — volume mensal da Ouvidoria de Poá

## Contexto e método

**Pergunta:** o histórico mensal ajuda a estimar o volume de manifestações registradas da Ouvidoria de Poá? O uso possível é explorar o volume agregado para orientar quando reforçar informação pública. A série não identifica qual serviço gerou a demanda e não sustenta uma previsão operacional.

A fonte primária é o Relatório Anual da Ouvidoria de Poá de 2024, que publica os valores mensais de 2023 e 2024. São 24 observações reais. Janeiro de 2023 fica fora da matriz supervisionada porque não há mês anterior para a defasagem; restam 23 linhas, com mês cíclico (seno e cosseno), ano e manifestações do mês anterior.

A divisão preserva a ordem temporal: 18 linhas para treino e os últimos 5 meses para teste. A avaliação prevê um mês à frente e, a cada mês de teste, usa a contagem real do mês imediatamente anterior como defasagem observada. A busca avalia 18 combinações de três hiperparâmetros com três folds de janela expansiva no treino; o teste não participa da seleção. O critério de seleção é MAE médio. A comparação é com a previsão ingênua do último mês. MAE, RMSE e R² são apresentados para treino e teste.

O modelo é um Random Forest de regressão didático e autocontido, implementado em experimento_ouvidoria.py com bootstrap e árvores CART. O código não usa scikit-learn, pois o ambiente desta execução não tinha esse pacote nem acesso ao PyPI. A implementação e essa limitação ficam explícitas para revisão.

## Dados e controle de qualidade

In [1]:
import csv
import sys
from pathlib import Path

ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "ml/dados/ouvidoria_poa_mensal.csv").is_file()
)
sys.path.insert(0, str(ROOT / "ml"))
from experimento_ouvidoria import (
    chronological_holdout, grid_search, load_monthly_data, metrics,
    prepare_supervised_rows, print_results, run_experiment,
    validate_monthly_data,
)

monthly = load_monthly_data(ROOT / "ml/dados/ouvidoria_poa_mensal.csv")
quality = validate_monthly_data(monthly)
print("Validação da série:", quality)
print("Primeira linha:", monthly[0])
print("Última linha:", monthly[-1])

Validação da série: {'linhas': 24, 'meses_consecutivos': True, 'duplicatas': 0, 'lacunas': 0, 'totais_anuais': {2023: 717, 2024: 853}}
Primeira linha: {'ano': 2023, 'mes': 1, 'manifestacoes_registradas': 63, 'url_fonte': 'https://poa.sp.gov.br/wp-content/uploads/2025/03/Relatorio-Ouvidoria-2024.pdf'}
Última linha: {'ano': 2024, 'mes': 12, 'manifestacoes_registradas': 24, 'url_fonte': 'https://poa.sp.gov.br/wp-content/uploads/2025/03/Relatorio-Ouvidoria-2024.pdf'}


## Busca e avaliação cronológica

A busca usa n_estimators = 50, 100 ou 200; max_depth = 2, 4 ou ilimitado; e min_samples_leaf = 1 ou 2. A tabela completa com as 18 combinações e os erros em cada fold também é salva em resultados/busca_hiperparametros.csv.

In [2]:
result = run_experiment(ROOT)
print_results(result)

print("\n18 combinações avaliadas (ordenadas pelo MAE médio de CV):")
with (ROOT / "ml/resultados/busca_hiperparametros.csv").open(
    encoding="utf-8", newline=""
) as source:
    for row in csv.DictReader(source):
        print(row)

Observações: 24 brutas; 23 após defasagem
Treino: 18 (02/2023 a 07/2024); teste: 5 (08/2024 a 12/2024)
Busca: 18 combinações; TimeSeriesSplit(3) somente no treino
Melhores parâmetros: {"n_estimators": 200, "max_depth": 2, "min_samples_leaf": 2}; MAE CV=21.82
Modelo | partição | MAE | RMSE | R²
Random Forest | treino | 9.72 | 13.22 | 0.68
Random Forest | teste | 17.01 | 18.41 | -0.06
Último mês | treino | 18.61 | 24.27 | -0.07
Último mês | teste | 18.00 | 22.86 | -0.63
Mês | real | Random Forest | último mês
08/2024 | 72 | 53.76 | 44
09/2024 | 64 | 54.75 | 72
10/2024 | 71 | 53.59 | 64
11/2024 | 65 | 54.06 | 71
12/2024 | 24 | 53.21 | 65

18 combinações avaliadas (ordenadas pelo MAE médio de CV):
{'n_estimators': '200', 'max_depth': '2', 'min_samples_leaf': '2', 'mae_fold_1': '19.661771', 'mae_fold_2': '31.778146', 'mae_fold_3': '14.020517', 'mae_cv_media': '21.820145'}
{'n_estimators': '200', 'max_depth': '4', 'min_samples_leaf': '2', 'mae_fold_1': '19.661771', 'mae_fold_2': '31.701604',

## TL;DR

<!-- RESULT_SUMMARY_BEGIN -->
Com 18 linhas de treino e 5 de teste (08/2024 a 12/2024), o Random Forest teve MAE de 17.01, RMSE de 18.41 e R² de -0.06. O baseline do último mês teve MAE de 18.00; o modelo reduziu o MAE em 5.5% frente ao último mês. Esse resultado descreve apenas os cinco meses reservados e não demonstra capacidade de previsão operacional.
<!-- RESULT_SUMMARY_END -->

## Resultados e figuras

A linha vertical marca o início do teste. Os gráficos são gerados pelo notebook e ficam disponíveis em SVG.

### Série observada e divisão

![Série mensal e separação treino/teste](../figuras/serie_mensal.svg)

### Teste: realizado, floresta e baseline

![Previsões no conjunto de teste](../figuras/previsao_teste.svg)

### Busca de hiperparâmetros

![Seis melhores combinações por MAE de validação](../figuras/cv_melhores.svg)

## Interpretação e limites

<!-- TAKEAWAYS_BEGIN -->
No modelo ajustado, o MAE de treino foi 9.72 e o de teste 17.01 (diferença teste menos treino: 7.29). Essa diferença é um sinal descritivo de ajuste, não um diagnóstico estável de sobreajuste com apenas cinco meses de teste.
<!-- TAKEAWAYS_END -->

- A amostra de teste tem somente cinco pontos; métricas variam bastante com um único mês e não comprovam generalização.
- O total reúne manifestações de temas diferentes e pode refletir mudanças de registro ou atendimento. Não usar o resultado para priorizar um serviço específico.
- O experimento é acadêmico e exploratório. O modelo não é integrado à API, ao painel ou ao aplicativo.

## Fontes

- [Relatório Anual da Ouvidoria de Poá 2024 (PDF)](https://poa.sp.gov.br/wp-content/uploads/2025/03/Relatorio-Ouvidoria-2024.pdf), tabela mensal, página 9.
- [Relatório Anual da Ouvidoria de Poá 2025 (PDF)](https://poa.sp.gov.br/wp-content/uploads/2026/04/Relatorio-Ouvidoria-2025.pdf), usado somente para contextualizar o total anual de 1.074 manifestações e distinguir as 557 ligações sem protocolo; seus dados não entram no treino nem no teste.
- Leia dados/README.md para extração, reconciliação e limitações.